In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
dataset = pd.read_csv('CKD.csv')

In [4]:
indep=dataset.drop(columns=['classification'])
dep=dataset['classification']
indep = pd.get_dummies(indep, dtype=int, drop_first=True)

In [5]:
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [7]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [8]:
from sklearn.linear_model import LogisticRegression

In [9]:
from sklearn.model_selection import GridSearchCV
param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 
grid = GridSearchCV(LogisticRegression(),param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='accuracy')
grid.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,estimator,LogisticRegression()
,param_grid,"{'penalty': ['l2'], 'solver': ['newton-cg', 'lbfgs', ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [10]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test)

In [11]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions) 

In [12]:
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [13]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'penalty': 'l2', 'solver': 'newton-cg'}: 0.9924946382275899


In [14]:
print(cm)

[[51  0]
 [ 1 81]]


In [15]:
print(clf_report)

              precision    recall  f1-score   support

          no       0.98      1.00      0.99        51
         yes       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [16]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[51  0]
 [ 1 81]]


In [17]:
table=pd.DataFrame.from_dict(re)

In [18]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_penalty,param_solver,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.016916,0.004357,0.004131,0.000902,l2,newton-cg,"{'penalty': 'l2', 'solver': 'newton-cg'}",0.981481,0.981132,0.981132,1.000000,1.000000,0.988749,0.009187,1
1,0.014703,0.003626,0.004505,0.001655,l2,lbfgs,"{'penalty': 'l2', 'solver': 'lbfgs'}",0.981481,0.981132,0.981132,1.000000,1.000000,0.988749,0.009187,1
2,0.006210,0.002448,0.003222,0.000702,l2,liblinear,"{'penalty': 'l2', 'solver': 'liblinear'}",0.962963,0.981132,0.962264,0.981132,0.981132,0.973725,0.009075,4
3,0.020729,0.004315,0.003418,0.000681,l2,saga,"{'penalty': 'l2', 'solver': 'saga'}",0.981481,0.981132,0.981132,0.981132,0.981132,0.981202,0.000140,3


In [19]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

          no       0.98      1.00      0.99        51
         yes       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [20]:
from sklearn.metrics import roc_auc_score
roc_auc=  roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])
print("ROC AUC Score of Logistic:", roc_auc)

ROC AUC Score of Logistic: 1.0
